# Covariance structure analysis across genomic foundation models

Loads cached `cov_inv` matrices from `run_metrics_only.ipynb` and asks:

1. **Eigenvalue spectrum** — How efficiently does each model use its embedding space?
2. **Effective dimensionality** — How many dimensions carry null variation?
3. **Distance-dependent covariance shift** — Do long-context models show distance-dependent null structure?
4. **Precision matrix structure** — What conditional dependencies exist between embedding dimensions?
5. **Cross-model covariance alignment** — Do architecturally similar models share null covariance structure?

In [ ]:
# ---------------------------------------------------------------------------
# Cell 1: Load all cached covariance matrices
# ---------------------------------------------------------------------------
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy import linalg
from collections import defaultdict

matplotlib.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "figure.facecolor": "white",
})

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "notebooks" / "paper_data_config.py").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.paper_data_config import embeddings_dir
from notebooks.processing.process_epistasis import FULL_MODEL_CONFIG

OUTPUT_BASE = embeddings_dir()
CACHE_DIR = OUTPUT_BASE / "cache"
FIG_DIR = OUTPUT_BASE / "figures" / "covariance"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Model metadata for plotting
MODEL_META = {}
for mk, (ctx, _) in FULL_MODEL_CONFIG.items():
    if mk.startswith("nt"):
        arch = "Transformer (NT)"
    elif mk in ("hyenadna",):
        arch = "SSM (HyenaDNA)"
    elif mk in ("caduceus",):
        arch = "SSM (Caduceus)"
    elif mk in ("evo2",):
        arch = "SSM (Evo2)"
    elif mk in ("borzoi",):
        arch = "CNN+Transformer (Borzoi)"
    elif mk in ("alphagenome",):
        arch = "Transformer (AlphaGenome)"
    elif mk in ("convnova",):
        arch = "CNN (ConvNova)"
    elif mk in ("mutbert",):
        arch = "Transformer (MutBERT)"
    elif mk in ("dnabert",):
        arch = "Transformer (DNABERT)"
    elif mk in ("rinalmo",):
        arch = "Transformer (RiNALMo)"
    elif mk in ("specieslm",):
        arch = "Transformer (SpeciesLM)"
    elif mk in ("spliceai",):
        arch = "CNN (SpliceAI)"
    else:
        arch = "Other"
    MODEL_META[mk] = {"context": ctx, "arch": arch}

# Architecture family for coloring (compatible with older matplotlib)
ARCH_FAMILIES = {
    "Transformer (NT)": "#1f77b4",          # blue
    "Transformer (MutBERT)": "#17becf",     # cyan
    "Transformer (DNABERT)": "#4682b4",     # steelblue
    "Transformer (RiNALMo)": "#00bfff",     # deepskyblue
    "Transformer (SpeciesLM)": "#4169e1",   # royalblue
    "Transformer (AlphaGenome)": "#000080",  # navy
    "SSM (HyenaDNA)": "#ff7f0e",            # orange
    "SSM (Caduceus)": "#d62728",            # red
    "SSM (Evo2)": "#8c564b",               # brown
    "CNN (ConvNova)": "#2ca02c",            # green
    "CNN (SpliceAI)": "#808000",            # olive
    "CNN+Transformer (Borzoi)": "#9467bd",  # purple
}

BIN_LABELS = ("0-1kb", "1-10kb", "10-100kb", "100kb+")

# Load global cov/cov_inv
global_cov = {}   # model_key -> cov (numpy)
global_inv = {}   # model_key -> cov_inv
for f in CACHE_DIR.glob("*_cov_inv.npz"):
    mk = f.stem.replace("_cov_inv", "")
    if mk in FULL_MODEL_CONFIG:
        data = np.load(f)
        global_cov[mk] = data["cov"]
        global_inv[mk] = data["cov_inv"]

# Load binned cov/cov_inv
binned_cov = defaultdict(dict)  # bin_label -> {model_key -> cov}
binned_inv = defaultdict(dict)
for label in list(BIN_LABELS) + ["global"]:
    safe = label.replace("+", "plus")
    for f in CACHE_DIR.glob(f"*_cov_inv_{safe}.npz"):
        mk = f.stem.replace(f"_cov_inv_{safe}", "")
        if mk in FULL_MODEL_CONFIG:
            data = np.load(f)
            binned_cov[label][mk] = data["cov"]
            binned_inv[label][mk] = data["cov_inv"]

MODEL_KEYS = sorted(global_cov.keys())
print(f"Loaded global cov for {len(MODEL_KEYS)} models: {MODEL_KEYS}")
print(f"Binned cov bins: {[k for k in binned_cov if k != 'global']}")
print(f"Figures will be saved to: {FIG_DIR}")

## 1. Eigenvalue spectrum — how efficiently does each model use its embedding space?

Models whose eigenvalues decay quickly concentrate null variation in few dimensions, meaning most of the embedding space is "quiet" under the null. Models with flat spectra spread variation uniformly — harder to separate signal from noise.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 2: Eigenvalue spectra — overlaid log-scale plot
# ---------------------------------------------------------------------------
eigenvalues = {}
for mk in MODEL_KEYS:
    eigs = np.linalg.eigvalsh(global_cov[mk])[::-1]  # descending
    eigenvalues[mk] = eigs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute eigenvalues (log scale)
ax = axes[0]
for mk in MODEL_KEYS:
    eigs = eigenvalues[mk]
    color = ARCH_FAMILIES.get(MODEL_META[mk]["arch"], "gray")
    ax.plot(np.arange(1, len(eigs) + 1), eigs, label=mk, color=color, alpha=0.8, linewidth=1.5)
ax.set_yscale("log")
ax.set_xlabel("Eigenvalue rank")
ax.set_ylabel("Eigenvalue (log scale)")
ax.set_title("Covariance eigenvalue spectrum")
ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.grid(True, alpha=0.3)

# Right: cumulative explained variance (normalized)
ax = axes[1]
for mk in MODEL_KEYS:
    eigs = eigenvalues[mk]
    cumvar = np.cumsum(eigs) / np.sum(eigs)
    color = ARCH_FAMILIES.get(MODEL_META[mk]["arch"], "gray")
    ax.plot(np.arange(1, len(eigs) + 1), cumvar, label=mk, color=color, alpha=0.8, linewidth=1.5)
ax.axhline(0.95, color="gray", linestyle="--", linewidth=0.8, label="95% variance")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("Cumulative variance explained")
ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "eigenvalue_spectra.png", bbox_inches="tight")
plt.show()

## 2. Effective dimensionality

**Participation ratio** `(sum λ)² / sum(λ²)` — number of dimensions that contribute meaningfully to null variance. A model with D-dimensional embeddings but effective rank 10 concentrates null variation in ~10 directions.

Also shows **dims for 95% variance** and **condition number** (ratio of largest to smallest eigenvalue — high condition number = the covariance is near-singular, Mahalanobis is unstable without regularization).

In [ ]:
# ---------------------------------------------------------------------------
# Cell 3: Effective dimensionality summary table + bar chart
# ---------------------------------------------------------------------------
rows = []
for mk in MODEL_KEYS:
    eigs = eigenvalues[mk]
    eigs_pos = eigs[eigs > 0]
    total_var = eigs_pos.sum()
    participation_ratio = total_var**2 / (eigs_pos**2).sum()
    cumvar = np.cumsum(eigs_pos) / total_var
    dims_95 = int(np.searchsorted(cumvar, 0.95)) + 1
    dims_99 = int(np.searchsorted(cumvar, 0.99)) + 1
    condition = eigs_pos[0] / eigs_pos[-1] if eigs_pos[-1] > 0 else float("inf")
    entropy = -np.sum((eigs_pos / total_var) * np.log(eigs_pos / total_var + 1e-30))
    rows.append({
        "model": mk,
        "embed_dim": len(eigs),
        "context": MODEL_META[mk]["context"],
        "arch": MODEL_META[mk]["arch"],
        "participation_ratio": round(participation_ratio, 1),
        "dims_95pct": dims_95,
        "dims_99pct": dims_99,
        "condition_number": f"{condition:.1e}",
        "spectral_entropy": round(entropy, 2),
    })

df_summary = pd.DataFrame(rows).sort_values("participation_ratio", ascending=False)
print(df_summary.to_string(index=False))

# Bar chart: participation ratio vs embedding dim
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_summary))
colors = [ARCH_FAMILIES.get(r["arch"], "gray") for _, r in df_summary.iterrows()]

bars_pr = ax.bar(x - 0.2, df_summary["participation_ratio"], 0.4, label="Effective rank (PR)", color=colors, alpha=0.85)
bars_dim = ax.bar(x + 0.2, df_summary["embed_dim"], 0.4, label="Embedding dim", color=colors, alpha=0.3)

ax.set_xticks(x)
ax.set_xticklabels(df_summary["model"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Dimensionality")
ax.set_title("Effective rank (participation ratio) vs embedding dimensionality")
ax.legend()
ax.grid(True, alpha=0.2, axis="y")

fig.tight_layout()
fig.savefig(FIG_DIR / "effective_dimensionality.png", bbox_inches="tight")
plt.show()

# Ratio plot: what fraction of dimensions are "used"?
fig, ax = plt.subplots(figsize=(10, 4))
df_summary["usage_ratio"] = df_summary["participation_ratio"] / df_summary["embed_dim"]
df_sorted = df_summary.sort_values("usage_ratio", ascending=True)
colors = [ARCH_FAMILIES.get(r["arch"], "gray") for _, r in df_sorted.iterrows()]
ax.barh(range(len(df_sorted)), df_sorted["usage_ratio"], color=colors, alpha=0.85)
ax.set_yticks(range(len(df_sorted)))
ax.set_yticklabels(df_sorted["model"], fontsize=8)
ax.set_xlabel("Effective rank / Embedding dim")
ax.set_title("Fraction of embedding space used for null variation")
ax.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
ax.grid(True, alpha=0.2, axis="x")

fig.tight_layout()
fig.savefig(FIG_DIR / "dimension_usage_ratio.png", bbox_inches="tight")
plt.show()

## 3. Distance-dependent covariance shift (receptive field effect)

**Key question:** Do long-context models (Borzoi 260kb, HyenaDNA 60kb) show different null covariance for nearby vs distant variant pairs?

If so, Mahalanobis-based epistasis detection requires distance-stratified calibration for these models. Short-context models (MutBERT 256bp, RiNALMo 511bp) should be distance-invariant because nearby pairs don't share more receptive field context.

We measure the **relative Frobenius distance** `||C_bin - C_global||_F / ||C_global||_F` — the fractional change in covariance structure per bin.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 4: Distance-dependent covariance shift
# ---------------------------------------------------------------------------
# Compute relative Frobenius distance between bin cov and global cov
shift_rows = []
for mk in MODEL_KEYS:
    if mk not in binned_cov.get("global", {}):
        continue
    cov_global = binned_cov["global"][mk]
    norm_global = np.linalg.norm(cov_global, "fro")
    for label in BIN_LABELS:
        if mk not in binned_cov.get(label, {}):
            continue
        cov_bin = binned_cov[label][mk]
        rel_frob = np.linalg.norm(cov_bin - cov_global, "fro") / norm_global
        # Also compute change in top eigenvalue
        eig_global = np.linalg.eigvalsh(cov_global)[-1]
        eig_bin = np.linalg.eigvalsh(cov_bin)[-1]
        top_eig_change = (eig_bin - eig_global) / eig_global
        shift_rows.append({
            "model": mk,
            "context": MODEL_META[mk]["context"],
            "arch": MODEL_META[mk]["arch"],
            "distance_bin": label,
            "rel_frobenius": rel_frob,
            "top_eig_change": top_eig_change,
        })

df_shift = pd.DataFrame(shift_rows)

if len(df_shift) > 0:
    # Heatmap: models x bins
    pivot = df_shift.pivot(index="model", columns="distance_bin", values="rel_frobenius")
    pivot = pivot.reindex(columns=list(BIN_LABELS))
    # Sort by context length
    ctx_order = {mk: MODEL_META[mk]["context"] for mk in pivot.index}
    pivot = pivot.loc[sorted(pivot.index, key=lambda m: ctx_order.get(m, 0), reverse=True)]

    fig, ax = plt.subplots(figsize=(8, max(4, len(pivot) * 0.45)))
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ctx_labels = [f"{mk} ({MODEL_META[mk]['context']//1000}kb)" for mk in pivot.index]
    ax.set_yticklabels(ctx_labels, fontsize=8)
    ax.set_title("Relative covariance shift by variant pair distance\n||C_bin - C_global||_F / ||C_global||_F")
    ax.set_xlabel("Variant pair distance bin")
    # Annotate cells
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                        color="white" if v > pivot.values[~np.isnan(pivot.values)].max() * 0.6 else "black")
    fig.colorbar(im, ax=ax, label="Relative Frobenius distance", shrink=0.8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "covariance_shift_heatmap.png", bbox_inches="tight")
    plt.show()

    # Line plot: shift vs distance bin, colored by context length
    fig, ax = plt.subplots(figsize=(10, 5))
    for mk in pivot.index:
        vals = pivot.loc[mk].values
        color = ARCH_FAMILIES.get(MODEL_META[mk]["arch"], "gray")
        ctx = MODEL_META[mk]["context"]
        ax.plot(range(len(BIN_LABELS)), vals, "o-", label=f"{mk} ({ctx//1000}kb)",
                color=color, alpha=0.8, linewidth=1.5, markersize=5)
    ax.set_xticks(range(len(BIN_LABELS)))
    ax.set_xticklabels(BIN_LABELS)
    ax.set_xlabel("Variant pair distance bin")
    ax.set_ylabel("Relative Frobenius distance from global")
    ax.set_title("Covariance shift: nearby pairs deviate more for long-context models")
    ax.legend(fontsize=7, ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "covariance_shift_lines.png", bbox_inches="tight")
    plt.show()
else:
    print("No binned covariance data available. Run run_metrics_only.ipynb cell 4 first.")

## 4. Precision matrix structure

The precision matrix (inverse covariance) reveals **conditional dependencies** between embedding dimensions. Off-diagonal entries `P_ij` are nonzero only when dimensions `i` and `j` are directly dependent (not mediated through other dimensions).

We visualize:
- Sorted precision matrices as heatmaps (clustered by correlation)
- Sparsity patterns (fraction of "near-zero" entries)
- Degree distribution of the conditional dependency graph

In [ ]:
# ---------------------------------------------------------------------------
# Cell 5: Precision matrix heatmaps (hierarchically clustered)
# ---------------------------------------------------------------------------
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

# Pick a subset of models to show (otherwise too many panels)
show_models = [mk for mk in MODEL_KEYS if not mk.startswith("nt") or mk in ("nt500_multi", "nt2500_multi")]
n_show = len(show_models)
ncols = min(4, n_show)
nrows = (n_show + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
axes = np.atleast_2d(axes)

for idx, mk in enumerate(show_models):
    ax = axes[idx // ncols, idx % ncols]
    P = global_inv[mk]
    d = P.shape[0]

    # Convert precision to partial correlation for better visualization
    # partial_corr(i,j) = -P_ij / sqrt(P_ii * P_jj)
    diag = np.sqrt(np.abs(np.diag(P)))
    diag[diag == 0] = 1
    pcorr = -P / np.outer(diag, diag)
    np.fill_diagonal(pcorr, 1.0)

    # Hierarchical clustering for ordering
    try:
        dist = 1 - np.abs(pcorr)
        np.fill_diagonal(dist, 0)
        dist = (dist + dist.T) / 2
        dist = np.clip(dist, 0, None)
        condensed = squareform(dist, checks=False)
        Z = linkage(condensed, method="ward")
        order = leaves_list(Z)
    except Exception:
        order = np.arange(d)

    pcorr_sorted = pcorr[np.ix_(order, order)]
    vmax = np.percentile(np.abs(pcorr_sorted[np.triu_indices(d, k=1)]), 99)

    im = ax.imshow(pcorr_sorted, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(f"{mk} ({d}d)", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

# Remove empty axes
for idx in range(n_show, nrows * ncols):
    axes[idx // ncols, idx % ncols].axis("off")

fig.suptitle("Partial correlation matrices (clustered)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "precision_heatmaps.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 6: Precision matrix sparsity analysis
# ---------------------------------------------------------------------------
# For each model, measure what fraction of partial correlations are "near zero"
# (below various thresholds). Sparser = more conditionally independent dimensions.

thresholds = [0.01, 0.05, 0.1]
sparsity_rows = []
for mk in MODEL_KEYS:
    P = global_inv[mk]
    d = P.shape[0]
    diag = np.sqrt(np.abs(np.diag(P)))
    diag[diag == 0] = 1
    pcorr = -P / np.outer(diag, diag)
    np.fill_diagonal(pcorr, 0)
    off_diag = np.abs(pcorr[np.triu_indices(d, k=1)])
    n_pairs = len(off_diag)

    row = {"model": mk, "arch": MODEL_META[mk]["arch"], "embed_dim": d,
           "mean_abs_pcorr": off_diag.mean(), "max_abs_pcorr": off_diag.max()}
    for t in thresholds:
        row[f"sparse_{t}"] = (off_diag < t).sum() / n_pairs
    sparsity_rows.append(row)

df_sparsity = pd.DataFrame(sparsity_rows).sort_values("mean_abs_pcorr")
print(df_sparsity[["model", "embed_dim", "mean_abs_pcorr", "max_abs_pcorr",
                     "sparse_0.01", "sparse_0.05", "sparse_0.1"]].to_string(index=False))

# Bar chart of mean absolute partial correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors = [ARCH_FAMILIES.get(r["arch"], "gray") for _, r in df_sparsity.iterrows()]
ax.barh(range(len(df_sparsity)), df_sparsity["mean_abs_pcorr"], color=colors, alpha=0.85)
ax.set_yticks(range(len(df_sparsity)))
ax.set_yticklabels(df_sparsity["model"], fontsize=8)
ax.set_xlabel("Mean |partial correlation|")
ax.set_title("Average conditional dependency strength")
ax.grid(True, alpha=0.2, axis="x")

ax = axes[1]
for t_idx, t in enumerate(thresholds):
    col = f"sparse_{t}"
    ax.barh(np.arange(len(df_sparsity)) + t_idx * 0.25 - 0.25,
            df_sparsity[col], height=0.25, alpha=0.8, label=f"|pcorr| < {t}")
ax.set_yticks(range(len(df_sparsity)))
ax.set_yticklabels(df_sparsity["model"], fontsize=8)
ax.set_xlabel("Fraction of dimension pairs")
ax.set_title("Precision matrix sparsity (fraction near-zero)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2, axis="x")

fig.tight_layout()
fig.savefig(FIG_DIR / "precision_sparsity.png", bbox_inches="tight")
plt.show()

## 5. Cross-model covariance alignment

Do architecturally similar models share the same null covariance structure? We compute the **RV coefficient** (multivariate generalization of R²) between every pair of models' covariance matrices and cluster them.

Models that cluster together have similar "noise geometry" — they organize null variation along similar directions, even if their embeddings differ.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 7: Cross-model covariance alignment (RV coefficient + eigenspectrum correlation)
# ---------------------------------------------------------------------------
from scipy.cluster.hierarchy import linkage, dendrogram

def rv_coefficient(A, B):
    """RV coefficient between two square matrices (same size required).
    Compares the structure by flattening and computing cosine similarity
    of the vectorized covariance matrices."""
    a = A.ravel()
    b = B.ravel()
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def eigenspectrum_correlation(A, B):
    """Pearson correlation between sorted eigenvalue spectra (same dim required)."""
    ea = np.sort(np.linalg.eigvalsh(A))[::-1]
    eb = np.sort(np.linalg.eigvalsh(B))[::-1]
    return np.corrcoef(ea, eb)[0, 1]

# Group models by embedding dimension (RV only makes sense for same-dim pairs)
dim_groups = defaultdict(list)
for mk in MODEL_KEYS:
    d = global_cov[mk].shape[0]
    dim_groups[d].append(mk)

print("Models grouped by embedding dimension:")
for d, mks in sorted(dim_groups.items()):
    print(f"  dim={d}: {mks}")

# For the largest group, compute full RV matrix
largest_group = max(dim_groups.values(), key=len)
if len(largest_group) >= 3:
    n = len(largest_group)
    rv_matrix = np.ones((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            rv = rv_coefficient(global_cov[largest_group[i]], global_cov[largest_group[j]])
            rv_matrix[i, j] = rv
            rv_matrix[j, i] = rv

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Heatmap
    ax = axes[0]
    im = ax.imshow(rv_matrix, cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(n))
    ax.set_xticklabels(largest_group, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n))
    ax.set_yticklabels(largest_group, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{rv_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if rv_matrix[i,j] < 0.5 else "black")
    fig.colorbar(im, ax=ax, label="RV coefficient", shrink=0.8)
    ax.set_title(f"RV coefficient (dim={global_cov[largest_group[0]].shape[0]})")

    # Dendrogram
    ax = axes[1]
    dist = 1 - rv_matrix
    np.fill_diagonal(dist, 0)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    dendrogram(Z, labels=largest_group, ax=ax, leaf_rotation=45, leaf_font_size=8)
    ax.set_ylabel("1 - RV coefficient")
    ax.set_title("Hierarchical clustering by covariance similarity")

    fig.tight_layout()
    fig.savefig(FIG_DIR / "cross_model_rv_coefficient.png", bbox_inches="tight")
    plt.show()
else:
    print("Largest same-dimension group has fewer than 3 models, skipping RV matrix.")

# Eigenspectrum correlation across ALL model pairs (works across different dims
# by comparing normalized cumulative variance curves resampled to common grid)
def normalized_cumvar_curve(cov, n_points=200):
    eigs = np.sort(np.linalg.eigvalsh(cov))[::-1]
    eigs_pos = eigs[eigs > 0]
    cumvar = np.cumsum(eigs_pos) / eigs_pos.sum()
    # Resample to n_points on [0, 1] x-axis (fraction of dimensions)
    x_orig = np.linspace(0, 1, len(cumvar))
    x_new = np.linspace(0, 1, n_points)
    return np.interp(x_new, x_orig, cumvar)

curves = {mk: normalized_cumvar_curve(global_cov[mk]) for mk in MODEL_KEYS}
n_all = len(MODEL_KEYS)
spec_corr = np.ones((n_all, n_all))
for i in range(n_all):
    for j in range(i + 1, n_all):
        r = np.corrcoef(curves[MODEL_KEYS[i]], curves[MODEL_KEYS[j]])[0, 1]
        spec_corr[i, j] = r
        spec_corr[j, i] = r

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Heatmap
ax = axes[0]
im = ax.imshow(spec_corr, cmap="viridis", vmin=spec_corr.min(), vmax=1, aspect="auto")
ax.set_xticks(range(n_all))
ax.set_xticklabels(MODEL_KEYS, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(n_all))
ax.set_yticklabels(MODEL_KEYS, fontsize=7)
fig.colorbar(im, ax=ax, label="Spectral shape correlation", shrink=0.8)
ax.set_title("Eigenspectrum shape similarity (all models)")

# Dendrogram
ax = axes[1]
dist_all = 1 - spec_corr
np.fill_diagonal(dist_all, 0)
dist_all = np.clip(dist_all, 0, None)
condensed_all = squareform(dist_all, checks=False)
Z_all = linkage(condensed_all, method="average")
dendrogram(Z_all, labels=MODEL_KEYS, ax=ax, leaf_rotation=45, leaf_font_size=7)
ax.set_ylabel("1 - spectral correlation")
ax.set_title("Clustering by eigenvalue decay shape")

fig.tight_layout()
fig.savefig(FIG_DIR / "eigenspectrum_similarity.png", bbox_inches="tight")
plt.show()

## 6. Distance-bin eigenspectrum shift (per model)

For each model, overlay the eigenvalue spectrum from each distance bin. If a model's receptive field is large enough to encompass both variants at close range, the close-range bin should show inflated top eigenvalues (more shared context = more correlated null residual dimensions).

In [ ]:
# ---------------------------------------------------------------------------
# Cell 8: Per-model eigenspectra across distance bins
# ---------------------------------------------------------------------------
models_with_bins = sorted(set.intersection(
    *[set(binned_cov.get(label, {}).keys()) for label in BIN_LABELS]
) if all(label in binned_cov for label in BIN_LABELS) else set())

if models_with_bins:
    n = len(models_with_bins)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes[np.newaxis, :]
    elif ncols == 1:
        axes = axes[:, np.newaxis]

    bin_colors = {"0-1kb": "tab:red", "1-10kb": "tab:orange", "10-100kb": "tab:blue", "100kb+": "tab:green"}

    for idx, mk in enumerate(models_with_bins):
        ax = axes[idx // ncols, idx % ncols]
        # Global spectrum
        eigs_g = np.sort(np.linalg.eigvalsh(binned_cov["global"][mk]))[::-1]
        ax.plot(range(1, len(eigs_g) + 1), eigs_g, "k-", alpha=0.4, linewidth=2, label="global")
        # Per-bin
        for label in BIN_LABELS:
            if mk in binned_cov.get(label, {}):
                eigs_b = np.sort(np.linalg.eigvalsh(binned_cov[label][mk]))[::-1]
                ax.plot(range(1, len(eigs_b) + 1), eigs_b, color=bin_colors[label],
                        alpha=0.8, linewidth=1.2, label=label)
        ax.set_yscale("log")
        ax.set_title(f"{mk} ({MODEL_META[mk]['context']//1000}kb ctx)", fontsize=9)
        ax.set_xlabel("Rank", fontsize=8)
        ax.set_ylabel("Eigenvalue", fontsize=8)
        ax.tick_params(labelsize=7)
        if idx == 0:
            ax.legend(fontsize=6, loc="upper right")
        ax.grid(True, alpha=0.2)

    for idx in range(n, nrows * ncols):
        axes[idx // ncols, idx % ncols].axis("off")

    fig.suptitle("Eigenvalue spectra by variant pair distance bin", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "eigenspectra_by_distance_bin.png", bbox_inches="tight")
    plt.show()
else:
    print("No models have all distance bins. Run run_metrics_only.ipynb cell 4 first.")

## 7. Context length vs covariance shift (the punchline)

Scatter plot: model context length (x) vs covariance shift at close range (y). If the receptive field hypothesis holds, this should show a positive correlation — long-context models have larger covariance shifts for nearby variant pairs.

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9: Context length vs covariance shift scatter
# ---------------------------------------------------------------------------
if len(df_shift) > 0:
    # Use the closest bin (0-1kb) as the "close range" shift
    df_close = df_shift[df_shift["distance_bin"] == "0-1kb"].copy()
    df_far = df_shift[df_shift["distance_bin"] == "100kb+"].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, df_plot, bin_label in [(axes[0], df_close, "0-1kb"), (axes[1], df_far, "100kb+")]:
        if len(df_plot) == 0:
            ax.set_title(f"No data for bin {bin_label}")
            continue

        for _, row in df_plot.iterrows():
            color = ARCH_FAMILIES.get(row["arch"], "gray")
            ax.scatter(row["context"], row["rel_frobenius"], c=color, s=80, alpha=0.85, edgecolors="black", linewidth=0.5)
            ax.annotate(row["model"], (row["context"], row["rel_frobenius"]),
                       fontsize=7, ha="left", va="bottom", xytext=(4, 2), textcoords="offset points")

        ax.set_xscale("log")
        ax.set_xlabel("Context length (bp, log scale)")
        ax.set_ylabel("Relative Frobenius distance from global")
        ax.set_title(f"Covariance shift at {bin_label} vs context length")
        ax.grid(True, alpha=0.3)

        # Compute and show Spearman correlation
        from scipy.stats import spearmanr
        rho, pval = spearmanr(df_plot["context"], df_plot["rel_frobenius"])
        ax.text(0.05, 0.95, f"Spearman r={rho:.2f}, p={pval:.3f}",
                transform=ax.transAxes, fontsize=9, va="top",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))

    fig.suptitle("Do long-context models need distance-stratified Mahalanobis calibration?", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "context_vs_covariance_shift.png", bbox_inches="tight")
    plt.show()

    # Summary table: shift ratio (close / far) per model
    if len(df_close) > 0 and len(df_far) > 0:
        merged = df_close[["model", "context", "arch", "rel_frobenius"]].merge(
            df_far[["model", "rel_frobenius"]], on="model", suffixes=("_close", "_far"))
        merged["shift_ratio"] = merged["rel_frobenius_close"] / merged["rel_frobenius_far"].clip(lower=1e-6)
        merged = merged.sort_values("shift_ratio", ascending=False)
        print("\nClose-range (0-1kb) vs far-range (100kb+) covariance shift ratio:")
        print(merged[["model", "context", "arch", "rel_frobenius_close", "rel_frobenius_far", "shift_ratio"]].to_string(index=False))
else:
    print("No distance-binned covariance data available.")

## Summary of saved figures

All figures saved to `{OUTPUT_BASE}/figures/covariance/`:

| File | What it shows |
|------|--------------|
| `eigenvalue_spectra.png` | Eigenvalue decay + cumulative variance for all models |
| `effective_dimensionality.png` | Participation ratio vs embedding dim |
| `dimension_usage_ratio.png` | Fraction of embedding space carrying null variation |
| `covariance_shift_heatmap.png` | How much covariance changes per distance bin per model |
| `covariance_shift_lines.png` | Same as above, as line plot |
| `precision_heatmaps.png` | Clustered partial correlation matrices |
| `precision_sparsity.png` | How sparse the conditional dependency structure is |
| `cross_model_rv_coefficient.png` | RV coefficient heatmap + dendrogram (same-dim models) |
| `eigenspectrum_similarity.png` | Spectral shape similarity across all models |
| `eigenspectra_by_distance_bin.png` | Per-model eigenspectra overlaid by distance bin |
| `context_vs_covariance_shift.png` | The punchline: context length predicts covariance shift |

## 8. Signal covariance: TCGA epistasis residuals vs null (1kGP)

The null covariance (above) captures where **population variation** lives. But where does the **epistatic signal** live? If signal and noise occupy different subspaces, Mahalanobis correction helps. If they overlap, it doesn't.

We compute the covariance of TCGA somatic epistasis residuals and compare:
1. **Eigenspectrum**: does TCGA signal concentrate in fewer dims than the null?
2. **Subspace overlap**: do the top eigenvectors of signal and null point in the same directions?
3. **Per-model comparison**: which models separate signal from noise most effectively?

In [ ]:
# ---------------------------------------------------------------------------
# Cell 10: Compute TCGA signal covariance from embedding DBs
# ---------------------------------------------------------------------------
# IMPORTANT: Ledoit-Wolf shrinkage depends on sample size — more samples = less
# shrinkage = more anisotropic estimate. To make null vs signal comparable,
# we subsample both to the SAME N before fitting covariance.
from genebeddings import VariantEmbeddingDB
from genebeddings.epistasis_features import _list_epistasis_ids_from_db, _load_residual_from_db, fit_covariance
import random

TCGA_SOURCE = "tcga_doubles"
RANDOM_STATE = 42

# First pass: load all residuals per source per model
raw_residuals = {}  # (source, model) -> list of residuals

for source in ["tcga_doubles", "fas_exon"]:
    for mk in MODEL_KEYS:
        db_path = OUTPUT_BASE / source / f"{mk}.db"
        if not db_path.exists():
            continue
        db = VariantEmbeddingDB(str(db_path))
        epi_ids = _list_epistasis_ids_from_db(db)
        residuals = []
        for eid in epi_ids:
            r = _load_residual_from_db(db, eid)
            if r is not None:
                residuals.append(r)
        db.close()
        if len(residuals) >= 50:
            raw_residuals[(source, mk)] = residuals
            print(f"  {source}/{mk}: {len(residuals)} residuals")

# Also need null residuals — reload from the okgp_chr12 source
for mk in MODEL_KEYS:
    key = ("okgp_chr12", mk)
    if key in raw_residuals:
        continue
    for src in ["okgp_chr12"]:
        db_path = OUTPUT_BASE / src / f"{mk}.db"
        if not db_path.exists():
            continue
        db = VariantEmbeddingDB(str(db_path))
        epi_ids = _list_epistasis_ids_from_db(db)
        residuals = []
        for eid in epi_ids:
            r = _load_residual_from_db(db, eid)
            if r is not None:
                residuals.append(r)
        db.close()
        if len(residuals) >= 50:
            raw_residuals[key] = residuals
            print(f"  {src}/{mk}: {len(residuals)} residuals")

# Second pass: for each model, subsample null and signal to same N, then fit
tcga_cov = {}; tcga_inv = {}; tcga_n_pairs = {}
fas_cov = {}
null_cov_matched = {}  # null covariance at matched sample size (for fair comparison)

rng = random.Random(RANDOM_STATE)

for mk in MODEL_KEYS:
    null_key = ("okgp_chr12", mk)
    tcga_key = ("tcga_doubles", mk)
    fas_key = ("fas_exon", mk)

    null_resids = raw_residuals.get(null_key, [])
    tcga_resids = raw_residuals.get(tcga_key, [])
    fas_resids = raw_residuals.get(fas_key, [])

    # TCGA vs null: subsample to min(N_tcga, N_null)
    if len(tcga_resids) >= 50 and len(null_resids) >= 50:
        n_match = min(len(tcga_resids), len(null_resids))
        tcga_sub = rng.sample(tcga_resids, n_match) if len(tcga_resids) > n_match else tcga_resids
        null_sub = rng.sample(null_resids, n_match) if len(null_resids) > n_match else null_resids

        cov_t, inv_t = fit_covariance(tcga_sub, method="ledoit_wolf", ridge=1e-6)
        cov_n, _ = fit_covariance(null_sub, method="ledoit_wolf", ridge=1e-6)

        tcga_cov[mk] = cov_t
        tcga_inv[mk] = inv_t
        tcga_n_pairs[mk] = n_match
        null_cov_matched[mk] = cov_n
        print(f"  {mk}: matched N={n_match} (TCGA had {len(tcga_resids)}, null had {len(null_resids)})")

    # FAS: subsample to match FAS size from null
    if len(fas_resids) >= 50 and len(null_resids) >= 50:
        n_match_fas = min(len(fas_resids), len(null_resids))
        fas_sub = rng.sample(fas_resids, n_match_fas) if len(fas_resids) > n_match_fas else fas_resids
        cov_f, _ = fit_covariance(fas_sub, method="ledoit_wolf", ridge=1e-6)
        fas_cov[mk] = cov_f

print(f"\nTCGA signal cov (N-matched): {len(tcga_cov)} models")
print(f"FAS signal cov: {len(fas_cov)} models")
print(f"Null cov (N-matched to TCGA): {len(null_cov_matched)} models")

In [ ]:
# ---------------------------------------------------------------------------
# Cell 11: Signal vs null eigenspectrum comparison
# ---------------------------------------------------------------------------
# Side-by-side: null (1kGP) vs signal (TCGA) eigenvalue decay per model

show_models_sig = [mk for mk in MODEL_KEYS if mk in tcga_cov and mk in global_cov]
if not show_models_sig:
    show_models_sig = sorted(tcga_cov.keys())[:12]
n_show = len(show_models_sig)
ncols = min(4, n_show)
nrows = max(1, (n_show + ncols - 1) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows))
if nrows == 1 and ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = axes[np.newaxis, :]

for idx, mk in enumerate(show_models_sig):
    ax = axes[idx // ncols, idx % ncols]

    # Null eigenvalues (normalized)
    eigs_null = np.sort(np.linalg.eigvalsh(global_cov[mk]))[::-1]
    eigs_null_norm = eigs_null / eigs_null.sum()

    # Signal eigenvalues (normalized)
    eigs_sig = np.sort(np.linalg.eigvalsh(tcga_cov[mk]))[::-1]
    eigs_sig_norm = eigs_sig / eigs_sig.sum()

    ax.plot(range(1, len(eigs_null_norm) + 1), eigs_null_norm,
            color="#1f77b4", alpha=0.8, linewidth=1.5, label="Null (1kGP)")
    ax.plot(range(1, len(eigs_sig_norm) + 1), eigs_sig_norm,
            color="#d62728", alpha=0.8, linewidth=1.5, label="Signal (TCGA)")

    # FAS if available
    if mk in fas_cov:
        eigs_fas = np.sort(np.linalg.eigvalsh(fas_cov[mk]))[::-1]
        eigs_fas_norm = eigs_fas / eigs_fas.sum()
        ax.plot(range(1, len(eigs_fas_norm) + 1), eigs_fas_norm,
                color="#2ca02c", alpha=0.7, linewidth=1.2, label="Signal (FAS)")

    ax.set_yscale("log")
    ax.set_title(f"{mk} ({tcga_n_pairs.get(mk, '?')} TCGA)", fontsize=9)
    ax.set_xlabel("Rank", fontsize=8)
    ax.set_ylabel("Normalized eigenvalue", fontsize=8)
    ax.tick_params(labelsize=7)
    if idx == 0:
        ax.legend(fontsize=6, loc="upper right")
    ax.grid(True, alpha=0.2)

for idx in range(n_show, nrows * ncols):
    axes[idx // ncols, idx % ncols].axis("off")

fig.suptitle("Eigenvalue spectra: null (1kGP) vs signal (TCGA / FAS)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "signal_vs_null_eigenspectra.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 12: Subspace overlap — do signal and null eigenvectors align?
# ---------------------------------------------------------------------------
# For each model, compute the cosine similarity between the top-k eigenvectors
# of the null covariance and the signal covariance.
# Low overlap = signal lives in different subspace = Mahalanobis helps.

def subspace_overlap(cov_a, cov_b, k=None):
    """
    Compute overlap between top-k eigenspaces of two covariance matrices.

    Returns:
        grassmann_sim: mean |cos(angle)| between corresponding eigenvectors (0=orthogonal, 1=aligned)
        projection_frac: fraction of cov_b's top-k variance explained by cov_a's top-k subspace
    """
    d = cov_a.shape[0]
    if k is None:
        k = min(d, 50)

    eigs_a, vecs_a = np.linalg.eigh(cov_a)
    eigs_b, vecs_b = np.linalg.eigh(cov_b)

    # Top-k eigenvectors (eigh returns ascending, so take last k)
    V_a = vecs_a[:, -k:]  # (d, k)
    V_b = vecs_b[:, -k:]  # (d, k)

    # Grassmann-like similarity: mean absolute cosine between subspaces
    # Project V_b onto V_a's subspace
    cos_matrix = np.abs(V_a.T @ V_b)  # (k, k) — absolute cosines
    # For each signal eigenvector, how much does it align with the best null eigenvector?
    max_alignment_per_signal = cos_matrix.max(axis=0)  # best match per signal vec
    grassmann_sim = float(max_alignment_per_signal.mean())

    # Projection fraction: how much of signal's top-k variance is captured by null's top-k subspace?
    eigs_b_topk = eigs_b[-k:][::-1]
    proj = V_a.T @ V_b  # (k, k)
    projected_var = np.sum((proj ** 2) * eigs_b_topk[np.newaxis, :], axis=1)
    total_var_b = eigs_b_topk.sum()
    projection_frac = float(projected_var.sum() / (total_var_b + 1e-30))

    return grassmann_sim, projection_frac


# Compute for all models
overlap_rows = []
for mk in sorted(tcga_cov.keys()):
    if mk not in global_cov:
        continue
    d = global_cov[mk].shape[0]
    for k in [10, 50, min(d, 200)]:
        if k > d:
            continue
        gs, pf = subspace_overlap(global_cov[mk], tcga_cov[mk], k=k)
        overlap_rows.append({
            "model": mk,
            "arch": MODEL_META.get(mk, {}).get("arch", "Other"),
            "d": d,
            "k": k,
            "grassmann_sim": round(gs, 4),
            "projection_frac": round(pf, 4),
            "signal": "TCGA",
        })
    # FAS overlap if available
    if mk in fas_cov:
        for k in [10, 50, min(d, 200)]:
            if k > d:
                continue
            gs, pf = subspace_overlap(global_cov[mk], fas_cov[mk], k=k)
            overlap_rows.append({
                "model": mk,
                "arch": MODEL_META.get(mk, {}).get("arch", "Other"),
                "d": d,
                "k": k,
                "grassmann_sim": round(gs, 4),
                "projection_frac": round(pf, 4),
                "signal": "FAS",
            })

df_overlap = pd.DataFrame(overlap_rows)

# Print summary at k=50
print("=" * 100)
print("SUBSPACE OVERLAP: null (1kGP) vs signal (TCGA/FAS) at k=50")
print("grassmann_sim: mean alignment of top-50 signal eigenvecs with top-50 null eigenvecs")
print("projection_frac: fraction of signal variance captured by null subspace")
print("LOW overlap = signal lives in different subspace = Mahalanobis helps")
print("=" * 100)

k50 = df_overlap[df_overlap["k"] == 50]
if len(k50) > 0:
    pivot_gs = k50.pivot(index="model", columns="signal", values="grassmann_sim")
    pivot_pf = k50.pivot(index="model", columns="signal", values="projection_frac")
    combined = pivot_gs.add_suffix("_align").join(pivot_pf.add_suffix("_proj"))
    print(combined.sort_values(combined.columns[0]).to_string())

# Bar chart: overlap per model
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, title in [
    (axes[0], "grassmann_sim", "Top-50 eigenvector alignment\n(1 = signal in noise subspace)"),
    (axes[1], "projection_frac", "Signal variance in null subspace\n(1 = fully captured by null)"),
]:
    tcga_k50 = df_overlap[(df_overlap["k"] == 50) & (df_overlap["signal"] == "TCGA")].sort_values(metric)
    colors = [ARCH_FAMILIES.get(MODEL_META.get(mk, {}).get("arch", ""), "gray") for mk in tcga_k50["model"]]
    y_pos = np.arange(len(tcga_k50))

    ax.barh(y_pos, tcga_k50[metric], color=colors, alpha=0.85, label="TCGA")

    # Overlay FAS if available
    fas_k50 = df_overlap[(df_overlap["k"] == 50) & (df_overlap["signal"] == "FAS")]
    if len(fas_k50) > 0:
        fas_vals = []
        for mk in tcga_k50["model"]:
            row = fas_k50[fas_k50["model"] == mk]
            fas_vals.append(row[metric].values[0] if len(row) > 0 else 0)
        ax.barh(y_pos + 0.3, fas_vals, height=0.3, color="#2ca02c", alpha=0.6, label="FAS")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(tcga_k50["model"], fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2, axis="x")

fig.suptitle("Signal-null subspace overlap (k=50)", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "signal_null_subspace_overlap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cell 13: Summary table — null vs signal effective dimensionality (N-matched)
# ---------------------------------------------------------------------------
# Uses null_cov_matched (subsampled to same N as TCGA) for fair comparison.

comp_rows = []
for mk in sorted(tcga_cov.keys()):
    # Use N-matched null if available, otherwise fall back to cached global
    null_mat = null_cov_matched.get(mk, global_cov.get(mk))
    if null_mat is None:
        continue
    d = null_mat.shape[0]

    for label, cov_mat in [("Null (1kGP, N-matched)", null_mat),
                            ("Null (1kGP, full)", global_cov.get(mk)),
                            ("TCGA", tcga_cov[mk])]:
        if cov_mat is None:
            continue
        eigs = np.linalg.eigvalsh(cov_mat)[::-1]
        eigs_pos = eigs[eigs > 0]
        total = eigs_pos.sum()
        pr = total**2 / (eigs_pos**2).sum()
        cumvar = np.cumsum(eigs_pos) / total
        d95 = int(np.searchsorted(cumvar, 0.95)) + 1
        comp_rows.append({
            "model": mk, "source": label, "d": d,
            "participation_ratio": round(pr, 1),
            "dims_95pct": d95,
            "usage_frac": round(pr / d, 4),
        })

    if mk in fas_cov:
        eigs = np.linalg.eigvalsh(fas_cov[mk])[::-1]
        eigs_pos = eigs[eigs > 0]
        total = eigs_pos.sum()
        pr = total**2 / (eigs_pos**2).sum()
        cumvar = np.cumsum(eigs_pos) / total
        d95 = int(np.searchsorted(cumvar, 0.95)) + 1
        comp_rows.append({
            "model": mk, "source": "FAS", "d": d,
            "participation_ratio": round(pr, 1),
            "dims_95pct": d95,
            "usage_frac": round(pr / d, 4),
        })

df_comp = pd.DataFrame(comp_rows)

print("=" * 100)
print("EFFECTIVE DIMENSIONALITY: null vs signal (N-matched for fair comparison)")
print(f"N per model = min(N_tcga, N_null) — see cell 10 for exact counts")
print("=" * 100)

# Show N-matched null vs TCGA (the fair comparison)
fair = df_comp[df_comp["source"].isin(["Null (1kGP, N-matched)", "TCGA", "FAS"])]
pivot_pr = fair.pivot(index="model", columns="source", values="participation_ratio")
pivot_uf = fair.pivot(index="model", columns="source", values="usage_frac")

display_df = pivot_pr.add_suffix("_PR").join(pivot_uf.add_suffix("_frac"))
display_df["d"] = [tcga_cov[mk].shape[0] for mk in display_df.index]
display_df["N_matched"] = [tcga_n_pairs.get(mk, "?") for mk in display_df.index]
cols = ["d", "N_matched"]
for src in ["Null (1kGP, N-matched)", "TCGA", "FAS"]:
    if f"{src}_PR" in display_df.columns:
        cols.extend([f"{src}_PR", f"{src}_frac"])
display_df = display_df[[c for c in cols if c in display_df.columns]]
print(display_df.sort_values("d").to_string())

# Also show the effect of N-matching on null PR
print("\n\nEffect of N-matching on null participation ratio:")
full_vs_matched = df_comp[df_comp["source"].isin(["Null (1kGP, full)", "Null (1kGP, N-matched)"])]
if len(full_vs_matched) > 0:
    pvt = full_vs_matched.pivot(index="model", columns="source", values="participation_ratio")
    if "Null (1kGP, full)" in pvt.columns and "Null (1kGP, N-matched)" in pvt.columns:
        pvt["ratio"] = pvt["Null (1kGP, N-matched)"] / pvt["Null (1kGP, full)"]
        print(pvt.sort_values("ratio").to_string())
        print(f"\nMean ratio: {pvt['ratio'].mean():.2f} (>1 means N-matching inflates PR via more shrinkage)")

# Key comparison: TCGA/Null at matched N
print("\n\nSignal / Null participation ratio (N-matched):")
null_col = "Null (1kGP, N-matched)"
if null_col in pivot_pr.columns and "TCGA" in pivot_pr.columns:
    ratio = pivot_pr["TCGA"] / pivot_pr[null_col]
    for mk in sorted(ratio.index):
        null_pr = pivot_pr.loc[mk, null_col]
        sig_pr = pivot_pr.loc[mk, "TCGA"]
        r = ratio[mk]
        interpret = "signal MORE concentrated" if r < 1 else "signal MORE spread" if r > 1 else "equal"
        print(f"  {mk:15s}: TCGA/Null = {r:.2f} ({interpret})  "
              f"[null PR={null_pr:.0f}, TCGA PR={sig_pr:.0f}, N={tcga_n_pairs.get(mk, '?')}]")